## Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

In [3]:
import os
from dotenv import load_dotenv
import glob

from langchain_classic import text_splitter
from openai import OpenAI

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login

### Connecting to OpenAI API and Hugging Face:

In [4]:
MODEL = 'gpt-4.1-nano'
db_name = 'vector_db'

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv('HF_TOKEN')

openai = OpenAI()
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 1. Dividing Documents into Chunks:

In [6]:
# Loading in Everything in The Knowledge Base Using Langchain's Loaders:

folders = glob.glob('knowledge-base/*')
documents = []

for folder in folders:

    # Setting up Document-Type as Folder Name:
    doc_type = os.path.basename(folder)

    # Directory Loader Instance:
    loader = DirectoryLoader(folder, glob= '**/*.md', loader_cls= TextLoader, loader_kwargs= {'encoding': 'utf-8'})

    # Loading Data:
    folder_docs = loader.load()

    # Adding Everything to One List:
    for doc in folder_docs:
        doc.metadata['doc_type']= doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents')

Loaded 76 documents


In [7]:
# Dividing Documents into Chunks:
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap= 200)
chunks = text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks.')

Divided into 413 chunks.


### 2. Making Vectors and Storing them in Chroma:

In [10]:
# Picking an Embedding Model:
embeddings = OpenAIEmbeddings(model= 'text-embedding-3-large')

# Wiping Database if Already Exists:
if os.path.exists(db_name):
    Chroma(persist_directory= db_name, embedding_function= embeddings).delete_collection()

# Creating Vector Database using Chroma:
vectorstore = Chroma.from_documents(documents= chunks,
                                    embedding= embeddings,
                                    persist_directory= db_name)

print(f'VectorStore Created with {vectorstore._collection.count()} documents.')

VectorStore Created with 413 documents.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [11]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature= 0, model= MODEL)

In [12]:
retriever.invoke('Who is Avery?')

[Document(id='e7d56bd1-0d48-4e5a-b92e-36027c9a3c84', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the

In [13]:
llm.invoke('Who is Avery?')

AIMessage(content="Could you please provide more context or specify which Avery you're referring to? There are many individuals and characters named Avery.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_a65955873c', 'id': 'chatcmpl-DUu3rdtyNM5wP4VH3y0YSlhGR7JcK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9135-b9fa-7b63-826c-233594f091ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning':